In [1]:
from symmer.operators.base import PauliwordOp, QuantumState
import h5py
import numpy as np 
from pyscf import scf
import math

In [2]:
chkfile = "O2.h5"

# load chemistry data
mol_load = scf.chkfile.load_mol(chkfile)
scf_load = scf.ROHF(mol_load) 
scf_load.__dict__.update(scf.chkfile.load(chkfile, 'scf')) 

# print chemistry details (note other info can be accessed in these objects see PySCF for further details.)
print(f"n spin orbitals: {2*mol_load.nao}")
print(f"basis: {mol_load.basis}")
xyz_str = mol_load.tostring(format="xyz")
print(f"geometry:\n\n{xyz_str}")
print(f"unit: {mol_load.unit}")
print(f"initaliziation: {scf_load.init_guess}")
E_rohf = scf_load.e_tot
print(f"e_rohf: {E_rohf}")

n spin orbitals: 36
basis: 6-31G
geometry:

2
XYZ from PySCF
O           0.00000000        0.00000000        0.00000000
O           0.00000000        0.00000000        2.55000000
unit: angstrom
initaliziation: minao
e_rohf: -149.43882641386773


In [3]:
serach_space = math.comb(mol_load.nao, mol_load.nelec[0]) * math.comb(mol_load.nao, mol_load.nelec[1]) 
serach_space


1547282880

In [4]:
with h5py.File("O2_data.h5", "r") as infile:
    # print(infile.keys())
    H = PauliwordOp(infile["H_symp"][()], infile["H_coeff"][()])

    hf_psi = QuantumState(infile["hf_state_mat"][()], infile["hf_coeff"][()])
    
    ci1_psi = QuantumState(infile["cisd_max_state_mat"][()], infile["cisd_max_state_coeff"][()])
    e_ci1 = infile["e_cisd_max"][()][0]

    ci2_psi = QuantumState(infile["cisdtq_max_state_state_mat"][()], infile["cisdtq_max_state_coeff"][()])
    e_ci2 = infile["e_cisdtq_max"][()][0]

In [5]:
E_hf = H.expval(hf_psi)
E_hf

np.complex128(-149.438826413868+0j)

In [6]:
print(f"energy ROHF: {E_hf.real:0.9f}")
print(f"energy ci1 : {e_ci1:0.9f}")
print(f"energy ci2 : {e_ci2:0.9f}")

energy ROHF: -149.438826414
energy ci1 : -149.582300577
energy ci2 : -149.654687887


In [7]:
ci1_psi.dagger * hf_psi

np.complex128(0.9586292701173829+0j)

In [9]:
ci1_hf = ci1_psi.dagger * hf_psi
assert ci1_hf.imag == 0
ci1_hf = ci1_hf.real

ci2_hf = ci2_psi.dagger * hf_psi
assert ci2_hf.imag == 0
ci2_hf = ci2_hf.real

ci1_hf, ci2_hf

(np.float64(0.9586292701173829), np.float64(0.04747945721174949))

In [10]:
overlap_ci1_hf = np.abs(ci1_hf)**2
overlap_ci2_hf = np.abs(ci2_hf)**2

In [11]:
print(f"|< CI1 | ROHF >|^2 = {overlap_ci1_hf:0.16f}")
print(f"|< CI2 | ROHF >|^2 = {overlap_ci2_hf:0.16f}")

|< CI1 | ROHF >|^2 = 0.9189700775257862
|< CI2 | ROHF >|^2 = 0.0022542988571224


In [12]:
ci2_sorted = ci2_psi.sort() # <-- sort state by coefficent magnitude (largest first)

ci2_sorted[:20] # <-for space only print first 20 terms

 0.552+0.000j |111111111011101110010000000000000000> +
-0.552+0.000j |111111111010111011010000000000000000> +
-0.282+0.000j |111111111001111011100000000000000000> +
 0.282+0.000j |111111111011011110100000000000000000> +
-0.239+0.000j |111111110111101110100000000000000000> +
 0.239+0.000j |111111110110111011100000000000000000> +
-0.136+0.000j |111111111111101110000000000000000000> +
 0.136+0.000j |111111111110111011000000000000000000> +
 0.083+0.000j |111111111010110111100000000000000000> +
-0.083+0.000j |111111111011101101100000000000000000> +
-0.054+0.000j |111111111011101110000100000000000000> +
 0.054+0.000j |111111111010111011000100000000000000> +
-0.050+0.000j |111111111011111010010000000000000000> +
 0.047+0.000j |111111111111111010000000000000000000> +
 0.046+0.000j |111111111011101110000000000000000100> +
-0.046+0.000j |111111111010111011000000000000000100> +
 0.032+0.000j |111111111001111011001000000000000000> +
-0.032+0.000j |111111111011011110001000000000000000> +
-0.031+0.0

In [13]:
# look over dominate ampliitude terms and measure energy

for i in range(2):
    psi = QuantumState(ci2_sorted[i].state_matrix, [1])
    print("psi:", psi)
    e_psi = H.expval(psi).real
    print(e_psi)
    psi_ci2 = psi.dagger * ci2_psi
    assert psi_ci2.imag ==0
    psi_ci2 = psi_ci2.real
    print(f" < psi | ci2_psi > = {psi_ci2:0.16f}")
    print(f"|< psi | ci2_psi >|^2 = {np.abs(psi_ci2)**2:0.16f}")
    print()


psi:  1.000+0.000j |111111111011101110010000000000000000>
-149.46164408206852
 < psi | ci2_psi > = 0.5522148827718463
|< psi | ci2_psi >|^2 = 0.3049412767547239

psi:  1.000+0.000j |111111111010111011010000000000000000>
-149.46164408513565
 < psi | ci2_psi > = -0.5522142973723327
|< psi | ci2_psi >|^2 = 0.3049406302224191



In [14]:
int("111111111011101110010000000000000000",2)

68647714816

In [18]:
Hamiltonian_dictionary = H.to_dictionary

terms = list(Hamiltonian_dictionary.items())[:10] #<-- for space only print first 10 terms (use dictionary for access to all)
for P, coeff in terms:
    print(coeff, P)

(-94.83523305419628+0j) IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(10.731109225149355+0j) ZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(10.731109225149355+0j) IZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(-4.752064419143859e-06+0j) XZXIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(-4.752064419143859e-06+0j) YZYIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(-4.752064419143859e-06+0j) IXZXIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(-4.752064419143859e-06+0j) IYZYIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(0.08588839133259377+0j) XZZZXIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(0.08588839133259377+0j) YZZZYIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
(0.08588839133259377+0j) IXZZZXIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
